In [1]:
import gymnasium as gym
import numpy as np
import cv2
from pyrep import PyRep
from pyrep.robots.mobiles.pioneer_p3dx import PioneerP3DX
from pyrep.objects.vision_sensor import VisionSensor
from pyrep.objects.proximity_sensor import ProximitySensor
from pyrep.objects.shape import Shape

class DribblingEnv(gym.Env):
    """
    Robot che deve andare dritto lungo un corridoio/strada,
    evitare ostacoli con il sensore di prossimità,
    e tornare in linea retta (reward sul percorso).
    """

    SCENE_PATH      = "env/dribbling_scene.ttt"
    IMG_SIZE        = 64
    MAX_STEPS       = 500
    DIST_GOAL       = 0.3    # metri — distanza per considerare il goal raggiunto
    COLLISION_DIST  = 0.15   # metri — soglia collisione
    MAX_SENSOR_DIST = 2.0    # metri — distanza massima sensore (fuori range)

    def __init__(self, headless=True):
        super().__init__()

        # ── Avvio CoppeliaSim ───────────────────────────────────────────
        self.pr = PyRep()
        self.pr.launch(self.SCENE_PATH, headless=headless)
        self.pr.start()

        # ── Oggetti scena ───────────────────────────────────────────────
        self.robot  = PioneerP3DX()
        self.camera = VisionSensor('Vision_sensor')   # montata sul robot, guarda avanti
        self.prox   = ProximitySensor('Proximity_sensor')  # frontale
        self.goal   = Shape('Goal')                   # oggetto marker fine percorso

        # Posizione iniziale del robot (salvata per il reset)
        self.start_pos = self.robot.get_position()
        self.start_ori = self.robot.get_orientation()

        # ── Spaces ──────────────────────────────────────────────────────
        # Observation: immagine appiattita (64x64x3) + dist_sensore + offset_laterale
        n_img = self.IMG_SIZE * self.IMG_SIZE * 3
        self.observation_space = gym.spaces.Box(
            low=0.0, high=1.0,
            shape=(n_img + 2,),   # +2 → [distanza_ostacolo, offset_laterale]
            dtype=np.float32
        )

        # Action: [velocità_sinistra, velocità_destra] in [-1, 1]
        self.action_space = gym.spaces.Box(
            low=-1.0, high=1.0,
            shape=(2,),
            dtype=np.float32
        )

        self.step_count = 0

    # ── Observation ─────────────────────────────────────────────────────
    def _get_obs(self):
        # CAMERA — esattamente come robot.camera.read_video_frame()
        frame = self.camera.capture_rgb()                    # (H, W, 3), float 0-1
        frame_uint8 = (frame * 255).astype(np.uint8)
        frame_small = cv2.resize(frame_uint8, (self.IMG_SIZE, self.IMG_SIZE))
        frame_flat  = frame_small.flatten().astype(np.float32) / 255.0

        # SENSORE PROSSIMITÀ — come robot.sensor.sub_distance()
        dist = self.prox.read()
        if dist == 0.0:
            dist = self.MAX_SENSOR_DIST  # nessun ostacolo = distanza massima

        # OFFSET LATERALE — quanto il robot è spostato dall'asse del percorso (X)
        robot_x   = self.robot.get_position()[0]
        start_x   = self.start_pos[0]
        lateral_offset = abs(robot_x - start_x)  # 0 = sulla retta, >0 = deviato

        # Normalizzazioni
        dist_norm   = np.clip(dist / self.MAX_SENSOR_DIST, 0.0, 1.0)
        offset_norm = np.clip(lateral_offset / 1.0, 0.0, 1.0)  # max 1m di offset

        return np.concatenate([frame_flat, [dist_norm, offset_norm]]).astype(np.float32)

    # ── Reward ──────────────────────────────────────────────────────────
    def _compute_reward(self):
        pos        = self.robot.get_position()
        goal_pos   = self.goal.get_position()
        dist_goal  = np.linalg.norm(np.array(pos[:2]) - np.array(goal_pos[:2]))

        dist_obst  = self.prox.read()
        if dist_obst == 0.0:
            dist_obst = self.MAX_SENSOR_DIST

        lateral_offset = abs(pos[0] - self.start_pos[0])
        collision  = dist_obst < self.COLLISION_DIST
        reached    = dist_goal  < self.DIST_GOAL

        # ── Composizione reward ─────────────────────────────────────────
        # +1 per ogni step sopravvissuto (incentiva ad andare avanti)
        r_survive  = +1.0

        # Reward principale: avanzamento verso il goal lungo l'asse Y
        # Più si avvicina al goal, meglio è
        r_progress = -dist_goal * 0.5

        # Penalità per stare fuori dalla retta (torna sulla strada!)
        # Questo è il reward "segui la strada"
        r_path     = -lateral_offset * 3.0

        # Penalità collisione
        r_collision = -50.0 if collision else 0.0

        # Bonus goal raggiunto
        r_goal      = +100.0 if reached else 0.0

        # Piccola penalità per step (incentiva ad arrivare veloce)
        r_time      = -0.1

        reward = r_survive + r_progress + r_path + r_collision + r_goal + r_time
        done   = collision or reached or (self.step_count >= self.MAX_STEPS)

        return float(reward), done

    # ── Step ────────────────────────────────────────────────────────────
    def step(self, action):
        # Scala le azioni in m/s (Pioneer accetta ~0-2 m/s)
        v_left  = float(action[0]) * 2.0
        v_right = float(action[1]) * 2.0
        self.robot.set_joint_target_velocities([v_left, v_right])

        self.pr.step()   # avanza di 1 frame simulazione
        self.step_count += 1

        obs            = self._get_obs()
        reward, done   = self._compute_reward()

        return obs, reward, done, False, {}

    # ── Reset ────────────────────────────────────────────────────────────
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.pr.stop()
        self.pr.start()
        self.robot.set_position(self.start_pos)
        self.robot.set_orientation(self.start_ori)
        self.step_count = 0
        return self._get_obs(), {}

    # ── Chiudi ───────────────────────────────────────────────────────────
    def close(self):
        self.pr.stop()
        self.pr.shutdown()


ImportError: cannot import name 'PyRep' from 'pyrep' (c:\Users\manuc\miniconda3\envs\aidrones\Lib\site-packages\pyrep\__init__.py)

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecTransposeImage
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback

# ── Crea ambiente ────────────────────────────────────────────────────────
def make_env():
    return DribblingEnv(headless=True)

env = DummyVecEnv([make_env])

# ── Callbacks ────────────────────────────────────────────────────────────
checkpoint_cb = CheckpointCallback(
    save_freq=10_000,
    save_path="./checkpoints/",
    name_prefix="ppo_manu"
)

eval_cb = EvalCallback(
    DummyVecEnv([make_env]),
    best_model_save_path="./models/",
    log_path="./logs/",
    eval_freq=20_000,
    n_eval_episodes=5,
    deterministic=True
)

# ── Modello PPO ──────────────────────────────────────────────────────────
model = PPO(
    "MlpPolicy",          # MlpPolicy perché obs è flat (img+sensori concatenati)
    env,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,        # un po' di entropia per esplorare
    verbose=1,
    tensorboard_log="./logs/ppo_manu"
)

# ── Training ─────────────────────────────────────────────────────────────
print("Inizio training PPO - Manu...")
model.learn(
    total_timesteps=300_000,
    callback=[checkpoint_cb, eval_cb]
)

model.save("models/model_manu_PPO")
print("Modello salvato in models/model_manu_PPO.zip")

env.close()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO

model = PPO.load("models/model_manu_PPO")
env   = DribblingEnv(headless=False)  # headless=False → vedi la simulazione

obs, _ = env.reset()
frames, rewards = [], []

for step in range(500):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, _, _ = env.step(action)

    # Salva frame come facevi con RoboMaster
    frame = env.camera.capture_rgb()
    frames.append(((frame * 255).astype('uint8'), step))
    rewards.append(reward)

    if done:
        print(f"Episodio terminato allo step {step} | Reward totale: {sum(rewards):.1f}")
        break

env.close()

# ── Griglia frame (identica al tuo script RoboMaster) ────────────────────
n, cols = len(frames), 8
rows = max(1, (n + cols - 1) // cols)
fig, axes = plt.subplots(rows, cols, figsize=(20, rows * 3))
axes = axes.flatten() if n > 1 else [axes]

for i, (img, step) in enumerate(frames):
    axes[i].imshow(img)
    axes[i].set_title(f"step {step}", fontsize=8)
    axes[i].axis("off")

for j in range(len(frames), len(axes)):
    axes[j].axis("off")

plt.suptitle(f"Test PPO Dribbling | {n} frame", fontsize=14)
plt.tight_layout()
plt.show()
